# Pre-flight v2: everything we check before the large-scale LCL runs

### What changed since v1 (run it in the same folder, so v1's results are reused)

1. **Professor's question: a plain neural network.** New model `flat_L{2,4,6}`: an ordinary MLP that sees the **whole slate** (all 5 items' features concatenated) with 2, 4 or 6 hidden layers, all at the same parameter count as self-attention. Nothing about sets or averages is built in. If depth alone were enough, the deep versions should recover A as well as the others.
2. **Fixed softmax model.** v1's single softmax head could not express LCL at all (its value was one number per item). The new `softmaxov_h{h}` passes each item's full features through the attention weights and reads them out against $x_i$. With uniform attention weights it *is* LCL, so softmax re-weighting is the only difference from the bilinear model.
3. **5k sessions added** to the data-size checks, because 2.5k turned out to be too little for self-attention.
4. **Resume per model:** only models that are not already saved get trained, so the v1 results in `preflight_outputs/` are reused and only the new models run.
5. **Positions shuffled when probing the flat MLP.** It is the only model that cares about item order, so the probe averages over positions for it.
6. **A frozen settings file** (`final_config.json`) is written at the end, for the cluster runs later.

# Pre-flight: everything we check before the large-scale LCL runs

**Why this notebook exists.** The big experiments will take many GPU hours. If something is wrong with the measuring tools, the training settings, or the amount of data, we would only find out after spending all of that time. This notebook checks every one of those things on a small scale first and ends with a **go / no-go table**.

| Part | What it checks | Why it matters |
|---|---|---|
| P1 | The probe reads A and θ correctly, at d = 6 **and** d = 16 | the sweeps go up to 16 features |
| P2 | Metric C (rebuilt) reads the context effect correctly | it is one of the three metrics in the report |
| P3 | Learning rate and weight decay for every model, chosen by **validation NLL only** | otherwise every sweep changes two things at once |
| P4 | The **null floor** (how big Â looks when the truth is A = 0) at small/large data and more features | tells us the smallest effect we can honestly claim at each setting |
| P4b | A plain network (flat MLP, 2/4/6 layers) against self-attention, mean-pool and the oracle | the professor's question: is depth alone enough? |
| P5 | The new attention models (bilinear, softmax) behave as the theory says, incl. the Eckart–Young ceiling | needed for the rank experiment (E3) |
| P6 | How many seeds each comparison really needs | replaces the guess of "100 seeds" |
| P7 | How long each run takes on the cluster | turns the plan into a time budget |
| P8 | Go / no-go table + a saved settings file for the cluster runs | the single thing to look at at the end |

**How to run.** Top to bottom, one cell at a time (VS Code on the cluster is fine). P3 and P4 are the long ones; they save after every run and skip what is already saved, so if the session dies just run the cell again. To run the whole notebook as a batch job instead:

```bash
jupyter nbconvert --to notebook --execute preflight_v2.ipynb --output preflight_v2_done.ipynb --ExecutePreprocessor.timeout=-1
```

Set the environment variable `PF_QUICK=1` for a tiny run that only checks the code works (numbers from a quick run mean nothing).

## Setup

All the knobs live here. `MAX_EPOCHS` is raised to 500 (from 300) because the oracle's best epoch reached ~300 in earlier runs. Long training sets are processed in **chunks that add up to exactly the full-batch gradient**, so the protocol is unchanged but 300k sessions still fit in GPU memory.

In [ ]:
import os, json, copy, time, math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=1.05)

QUICK   = os.environ.get("PF_QUICK", "0") == "1"
DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
OUT_DIR = os.environ.get("PF_OUT", "preflight_outputs")
os.makedirs(OUT_DIR, exist_ok=True)
TAG = "preflight-v1"

# reference world (same as the main notebook)
D_REF, RANK_REF, N_REF, A_SCALE, SLATE = 6, 2, (2000 if QUICK else 40000), 1.0, 5
N_VAL = N_EVAL = N_SHIFT = 500 if QUICK else 5000

# training protocol
MAX_EPOCHS = 30 if QUICK else 500
PATIENCE   = 10 if QUICK else 60
CHUNK      = 50000            # sessions per gradient chunk (sum of chunks = full-batch gradient)

# probe and metric C
PROBE_K_PER_D = 3 if QUICK else 10      # K = 10 d probe pairs
PROBE_N       = 100 if QUICK else 500   # slates per pair
N_BOOT        = 10 if QUICK else 200
C_TRIALS      = 500 if QUICK else 4000

print(f"device={DEVICE} | quick={QUICK} | out={OUT_DIR}")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))

## Library 1: ground truth, slates, choices, splits

Same generator as before, now for any number of features $d$:

$$u_i(S) = (\theta + A\,y_S)^\top x_i, \qquad y_S=\tfrac1{|S|}\sum_{j\in S}x_j$$

`A_scale = 0` gives the **no-effect world** (same θ, same slates, A = 0).

In [ ]:
def make_ground_truth(d, rank, A_scale, seed):
    rng = np.random.default_rng(seed)
    theta = rng.normal(size=d); theta /= np.linalg.norm(theta)
    U, V = rng.normal(size=(d, rank)), rng.normal(size=(d, rank))
    A = U @ V.T
    return theta, A_scale * A / np.linalg.norm(A)

def simulate_choices(theta, A, X, seed):
    rng = np.random.default_rng(seed)
    util = np.einsum("nd,nsd->ns", theta[None, :] + X.mean(axis=1) @ A.T, X)
    z = util - util.max(axis=1, keepdims=True)
    probs = np.exp(z); probs /= probs.sum(axis=1, keepdims=True)
    choices = np.clip((rng.random((len(X), 1)) > probs.cumsum(axis=1)).sum(axis=1), 0, X.shape[1] - 1)
    return probs, choices

def build_splits(theta, A, n_train, d, data_seed):
    rng = np.random.default_rng(data_seed)
    X_all = rng.normal(size=(n_train + N_VAL + N_EVAL, SLATE, d))
    pool = np.random.default_rng(data_seed + 1).normal(size=(N_SHIFT * 20, SLATE, d))
    X_shift = pool[np.argsort(np.linalg.norm(pool.mean(axis=1), axis=1))[-N_SHIFT:]]
    parts = {"train": X_all[:n_train], "val": X_all[n_train:n_train + N_VAL],
             "eval_in": X_all[n_train + N_VAL:], "eval_shift": X_shift}
    out = {}
    for k, (name, Xp) in enumerate(parts.items()):
        probs, choices = simulate_choices(theta, A, Xp, seed=data_seed * 10 + k)
        out[name] = dict(X=Xp, probs=probs, choices=choices)
    return out

def to_dev(split):
    return dict(X=torch.tensor(split["X"], dtype=torch.float32, device=DEVICE),
                y=torch.tensor(split["choices"], dtype=torch.long, device=DEVICE),
                p=torch.tensor(split["probs"], dtype=torch.float32, device=DEVICE))

theta_chk, A_chk = make_ground_truth(D_REF, RANK_REF, A_SCALE, seed=12345)
print("check: ||A||_F =", round(np.linalg.norm(A_chk), 4), "| rank =", np.linalg.matrix_rank(A_chk))

## Library 2: the model ladder

| name | utility | role |
|---|---|---|
| `flat_L{L}` | MLP$([x_1;\dots;x_5])$ with L hidden layers, one score per position | a plain network that sees everything, no structure built in (professor's question) |
| `mean_pool` | MLP$([x_i;\,y_S])$ | correct inputs, averaging forced |
| `bilinear_h{h}` | $w^\top x_i + \frac1m\sum_j (W_Qx_i)\cdot(W_Kx_j)$ | **exactly LCL with rank(A) ≤ h**; the LCL twin of the CDM "raw" model |
| `softmaxov_h{h}` | $w^\top x_i + (W_Ox_i)^\top W_V \sum_j \mathrm{softmax}_j(q_i\cdot k_j/\sqrt h)\,x_j$ | one softmax head; with uniform weights it equals LCL with $A=W_O^\top W_V$ (rank ≤ h) |
| `self_attn_L{L}` | embed → L transformer layers (pre-LN) → linear score | the subject |
| `oracle` | $(\theta + A y_S)^\top x_i$ with free θ, A | the ceiling |

**Change from the main notebook:** the transformer now uses **pre-LN** (`norm_first=True`, plus a final LayerNorm) at every depth. Post-LN is known to train unstably at 6+ layers, and we want *one* architecture across the depth sweep, so the 1-layer model uses it too.

In [ ]:
class MeanPoolNetwork(nn.Module):
    def __init__(self, d, hidden=87):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(2 * d, hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 1))
    def forward(self, X):
        y = X.mean(dim=1, keepdim=True).expand(-1, X.shape[1], -1)
        return self.net(torch.cat([X, y], dim=-1)).squeeze(-1)

class SelfAttentionModel(nn.Module):
    def __init__(self, d, d_model=32, n_heads=2, n_layers=1):
        super().__init__()
        self.embed = nn.Linear(d, d_model)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads, dim_feedforward=2 * d_model,
                                           dropout=0.0, batch_first=True, norm_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers, norm=nn.LayerNorm(d_model),
                                             enable_nested_tensor=False)
        self.head = nn.Linear(d_model, 1)
    def forward(self, X):
        return self.head(self.encoder(self.embed(X))).squeeze(-1)

class BilinearAttention(nn.Module):
    # LCL written as attention: no softmax, plain average over ALL items (y_S includes x_i)
    def __init__(self, d, h):
        super().__init__()
        self.w = nn.Parameter(torch.zeros(d))
        self.Wq = nn.Parameter(torch.randn(h, d) * 0.1)
        self.Wk = nn.Parameter(torch.randn(h, d) * 0.1)
    def forward(self, X):
        q, k = X @ self.Wq.T, X @ self.Wk.T
        return X @ self.w + torch.einsum("nih,njh->ni", q, k) / X.shape[1]

class SoftmaxOV(nn.Module):
    # one softmax head; the attended FEATURES are read out against x_i.
    # q = k = 0 gives uniform weights, and then this is exactly LCL with A = Wo^T Wv (rank <= h)
    def __init__(self, d, h):
        super().__init__()
        self.w = nn.Parameter(torch.zeros(d))
        self.Wq, self.Wk = nn.Linear(d, h, bias=False), nn.Linear(d, h, bias=False)
        self.Wv, self.Wo = nn.Linear(d, h, bias=False), nn.Linear(d, h, bias=False)
        for lin in (self.Wq, self.Wk, self.Wv, self.Wo): nn.init.normal_(lin.weight, std=0.1)
        self.h = h
    def forward(self, X):
        att = torch.softmax(self.Wq(X) @ self.Wk(X).transpose(1, 2) / math.sqrt(self.h), dim=-1)
        ctx = att @ self.Wv(X)                                   # (n, m, h)
        return X @ self.w + (self.Wo(X) * ctx).sum(-1)

class FlatMLP(nn.Module):
    # a plain network: all items' features concatenated, one score per position out
    def __init__(self, d, n_layers, width):
        super().__init__()
        layers, inp = [], SLATE * d
        for _ in range(n_layers):
            layers += [nn.Linear(inp, width), nn.ReLU()]; inp = width
        layers.append(nn.Linear(inp, SLATE))
        self.net = nn.Sequential(*layers)
    def forward(self, X):
        return self.net(X.reshape(X.shape[0], -1))

_FLAT_WIDTH = {}
def flat_width(d, n_layers):
    # width that puts the flat MLP at the same parameter count as 1-layer self-attention
    if (d, n_layers) not in _FLAT_WIDTH:
        target = sum(p.numel() for p in SelfAttentionModel(d).parameters())
        cost = lambda w: (SLATE * d) * w + w + (n_layers - 1) * (w * w + w) + w * SLATE + SLATE
        _FLAT_WIDTH[(d, n_layers)] = min(range(4, 600), key=lambda w: abs(cost(w) - target))
    return _FLAT_WIDTH[(d, n_layers)]

class LCLOracle(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.theta = nn.Parameter(torch.randn(d) * 0.1)
        self.A = nn.Parameter(torch.randn(d, d) * 0.1)
    def forward(self, X):
        return torch.einsum("nd,nsd->ns", self.theta.unsqueeze(0) + X.mean(dim=1) @ self.A.T, X)

def build_model(name, d):
    if name == "mean_pool":             return MeanPoolNetwork(d)
    if name.startswith("self_attn_L"):  return SelfAttentionModel(d, n_layers=int(name[len("self_attn_L"):]))
    if name.startswith("bilinear_h"):   return BilinearAttention(d, int(name[len("bilinear_h"):]))
    if name.startswith("softmaxov_h"):  return SoftmaxOV(d, int(name[len("softmaxov_h"):]))
    if name.startswith("flat_L"):
        L = int(name[len("flat_L"):]);  return FlatMLP(d, L, flat_width(d, L))
    if name == "oracle":                return LCLOracle(d)
    raise ValueError(name)

def family(name):
    # models that share tuned hyperparameters (bilinear / softmaxov share across head sizes)
    for f in ["bilinear", "softmaxov"]:
        if name.startswith(f): return f
    return name

def is_equivariant(name):
    return not name.startswith("flat")          # only the flat MLP cares about item order

def n_params(m): return sum(p.numel() for p in m.parameters())

for nm in ["flat_L2", "flat_L4", "flat_L6", "mean_pool", "self_attn_L1", "self_attn_L6",
           f"bilinear_h{D_REF}", f"softmaxov_h{D_REF}", "oracle"]:
    extra = f"  (width {flat_width(D_REF, int(nm[6:]))})" if nm.startswith("flat") else ""
    print(f"{nm:<16}{n_params(build_model(nm, D_REF)):>8} parameters at d={D_REF}{extra}")

## Library 3: training and the predictive metrics

Full-batch AdamW, early stopping on **validation NLL only**. Every run also records:

- `budget_hit`: training ran out of epochs while still improving (then its "best" checkpoint is not real)
- `train_seconds`: for the time budget in P7
- KL(true ‖ model) on `eval_in` and `eval_shift` (exact, because the true choice probabilities are known)

In [ ]:
def full_batch_step(model, X, y, opt):
    opt.zero_grad()
    N, total = len(y), 0.0
    for i in range(0, N, CHUNK):                       # chunks add up to the exact full-batch gradient
        loss = F.cross_entropy(model(X[i:i + CHUNK]), y[i:i + CHUNK], reduction="sum") / N
        loss.backward(); total += loss.item()
    opt.step()
    return total

@torch.no_grad()
def eval_split(model, s):
    model.eval()
    nll = kl = 0.0
    N = len(s["y"])
    for i in range(0, N, CHUNK):
        logq = torch.log_softmax(model(s["X"][i:i + CHUNK]), dim=1)
        p = s["p"][i:i + CHUNK]
        nll += -logq.gather(1, s["y"][i:i + CHUNK, None]).sum().item()
        kl += (p * (torch.log(p + 1e-12) - logq)).sum().item()
    return nll / N, kl / N

def train_model(model, tr, va, lr, wd):
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    best, best_state, best_ep, t0 = float("inf"), None, 0, time.time()
    for ep in range(MAX_EPOCHS):
        model.train(); full_batch_step(model, tr["X"], tr["y"], opt)
        v = eval_split(model, va)[0]
        if v < best: best, best_ep, best_state = v, ep, copy.deepcopy(model.state_dict())
        if ep - best_ep >= PATIENCE: break
    model.load_state_dict(best_state)
    epochs = ep + 1
    return dict(epochs=epochs, best_epoch=best_ep, val_nll=best,
                budget_hit=bool(epochs == MAX_EPOCHS and best_ep > MAX_EPOCHS - PATIENCE),
                train_seconds=time.time() - t0)

def score_fn_of(model, shuffle=False, seed=0):
    # shuffle=True: every slate is shown to the model in a random item order and the scores are put back
    # in the original order. Changes nothing for order-free models; for the flat MLP it averages over positions.
    rng = np.random.default_rng(seed)
    @torch.no_grad()
    def run(X):
        model.eval()
        out = [model(torch.tensor(X[i:i + CHUNK], dtype=torch.float32, device=DEVICE)) for i in range(0, len(X), CHUNK)]
        return torch.cat(out).double().cpu().numpy()
    def f(X):
        if not shuffle: return run(X)
        perm = np.argsort(rng.random(X.shape[:2]), axis=1)
        sp = run(np.take_along_axis(X, perm[:, :, None], axis=1))
        s = np.empty_like(sp); np.put_along_axis(s, perm, sp, axis=1)
        return s
    return f

## Library 4: the probe and Metric C

**Probe** (unchanged from E0, now works for any $d$): fix a pair, vary the other items, fit the LCL line to the score gaps, all pairs jointly → $\hat A$, $\hat\theta$, fidelity, bootstrap CI, probe-invariance.

**Metric C, rebuilt.** It asks the behavioural question directly: *when only the surroundings change, does the model's preference between a and b move the way the true rule says?*

1. Draw a pair (a, b) and **two** ordinary filler sets, giving slates $S$ and $S'$ (both from the normal training distribution, no ±1.5 shifts).
2. Model's change: $\big[u_a-u_b\big]_{S'} - \big[u_a-u_b\big]_{S}$ (this is the change in $\log P(a)/P(b)$).
3. True change: $\delta^\top A\,(y_{S'}-y_S)$.
4. Over 4,000 such draws: **c_slope** (1 = right size), **c_r2** (shape), and **c_ratio** = RMS of the model's change ÷ RMS of the true change. In the no-effect world the true change is 0, so c_ratio there is the **false-positive size**, measured against the effect that the matching real A would produce.

In [ ]:
def run_probe(score_fn, d, seed=7, shifted=False, oversample=10):
    rng = np.random.default_rng(seed)
    K, d2 = PROBE_K_PER_D * d, d * d
    st = dict(G=np.zeros((K, d2, d2)), b=np.zeros((K, d2)), ss=np.zeros(K),
              mg=np.zeros(K), Yb=np.zeros((K, d)), Dl=np.zeros((K, d)))
    for k in range(K):
        x_a, x_b = rng.normal(size=(2, d))
        n_gen = PROBE_N * (oversample if shifted else 1)
        X = np.concatenate([np.broadcast_to(x_a, (n_gen, 1, d)), np.broadcast_to(x_b, (n_gen, 1, d)),
                            rng.normal(size=(n_gen, SLATE - 2, d))], axis=1)
        y = X.mean(axis=1)
        if shifted:
            keep = np.argsort(np.linalg.norm(y, axis=1))[-PROBE_N:]
            X, y = X[keep], y[keep]
        sc = score_fn(X); gap = sc[:, 0] - sc[:, 1]
        delta, y_bar, gap_bar = x_a - x_b, y.mean(axis=0), gap.mean()
        Z = np.einsum("p,nq->npq", delta, y - y_bar).reshape(len(y), d2)   # column p*d+q <-> A[p,q]
        g = gap - gap_bar
        st["G"][k], st["b"][k], st["ss"][k] = Z.T @ Z, Z.T @ g, g @ g
        st["mg"][k], st["Yb"][k], st["Dl"][k] = gap_bar, y_bar, delta
    return st

def fit_from_stats(st, idx):
    K, d = st["Dl"].shape
    w = np.bincount(idx, minlength=K).astype(float)          # bootstrap weights (1 for every pair = plain fit)
    G, b, ss = np.tensordot(w, st["G"], axes=1), w @ st["b"], w @ st["ss"]
    vecA = np.linalg.lstsq(G, b, rcond=None)[0]
    A_hat = vecA.reshape(d, d)
    Dl = st["Dl"][idx]
    c = st["mg"][idx] - np.einsum("kp,pq,kq->k", Dl, A_hat, st["Yb"][idx])
    th = np.linalg.lstsq(Dl, c, rcond=None)[0]
    fid_A = 1 - (ss - 2 * vecA @ b + vecA @ G @ vecA) / ss if ss > 1e-12 else np.nan
    res = c - Dl @ th
    fid_th = 1 - (res @ res) / (((c - c.mean()) ** 2).sum() + 1e-12)
    return A_hat, th, fid_A, fid_th

def recovery_metrics(A_hat, th_hat, A_true, th_true):
    a, b = A_hat.ravel(), A_true.ravel()
    fa, fb = np.linalg.norm(a), np.linalg.norm(b)
    out = dict(fro_hat=fa)
    if fb > 1e-12:
        out.update(cosine=a @ b / (fa * fb + 1e-12), rel_frob=np.linalg.norm(a - b) / fb, scale_ratio=fa / fb)
    else:
        out.update(cosine=np.nan, rel_frob=np.nan, scale_ratio=np.nan)
    out["theta_cos"] = th_hat @ th_true / (np.linalg.norm(th_hat) * np.linalg.norm(th_true) + 1e-12)
    out["theta_relerr"] = np.linalg.norm(th_hat - th_true) / np.linalg.norm(th_true)
    return out

def probe_report(score_fn, A_true, th_true, d, seed=7):
    st = run_probe(score_fn, d, seed=seed)
    K = len(st["ss"])
    A_hat, th_hat, fid_A, fid_th = fit_from_stats(st, np.arange(K))
    row = recovery_metrics(A_hat, th_hat, A_true, th_true)
    row.update(fid_A=fid_A, fid_theta=fid_th)
    brng = np.random.default_rng(seed + 999)
    boots = pd.DataFrame([recovery_metrics(*fit_from_stats(st, brng.integers(0, K, size=K))[:2], A_true, th_true)
                          for _ in range(N_BOOT)])
    for m in ["fro_hat", "cosine", "theta_cos"]:
        ok = boots[m].notna().any()
        row[m + "_lo"], row[m + "_hi"] = np.nanpercentile(boots[m], [2.5, 97.5]) if ok else (np.nan, np.nan)
    A_sh = fit_from_stats(run_probe(score_fn, d, seed=seed + 1, shifted=True), np.arange(K))[0]
    row["inv_abs"] = np.linalg.norm(A_hat - A_sh)
    row["inv_rel"] = row["inv_abs"] / (np.linalg.norm(A_hat) + 1e-12)
    return row, A_hat, th_hat

def metric_c(score_fn, A_true, A_ref, d, seed=11):
    rng = np.random.default_rng(seed)
    n = C_TRIALS
    xa, xb = rng.normal(size=(n, 1, d)), rng.normal(size=(n, 1, d))
    X1 = np.concatenate([xa, xb, rng.normal(size=(n, SLATE - 2, d))], axis=1)
    X2 = np.concatenate([xa, xb, rng.normal(size=(n, SLATE - 2, d))], axis=1)
    s1, s2 = score_fn(X1), score_fn(X2)
    d_mod = (s2[:, 0] - s2[:, 1]) - (s1[:, 0] - s1[:, 1])
    delta, dy = (xa - xb)[:, 0], X2.mean(axis=1) - X1.mean(axis=1)
    change = lambda A: np.einsum("np,pq,nq->n", delta, A, dy)
    rms = lambda v: float(np.sqrt(np.mean(v ** 2)))
    t, t_ref = change(A_true), change(A_ref)
    out = dict(c_ratio=rms(d_mod) / (rms(t_ref) + 1e-12))
    if np.std(t) > 1e-12:
        out["c_slope"] = float(np.cov(t, d_mod)[0, 1] / np.var(t, ddof=1))
        out["c_r2"] = float(np.corrcoef(t, d_mod)[0, 1] ** 2)
    else:
        out["c_slope"] = out["c_r2"] = np.nan
    return out

## Library 5: one runner for every experiment

`run_world` trains the requested models in one world (context or no-effect) for one configuration and seed, then measures everything. `run_jobs` loops over jobs, appends one CSV row per model, and **skips anything already saved**, so a long cell can be stopped and restarted.

In [ ]:
HP_PATH = os.path.join(OUT_DIR, "tuned_hyperparameters.json")
HP_DEFAULT = dict(lr=1e-2, wd=0.0)

def load_hp():
    if os.path.exists(HP_PATH):
        with open(HP_PATH) as f: return json.load(f)
    return {}

def run_world(cfg, world, seed, models, hp, measure=True):
    d, n, rank = cfg["d"], cfg["n"], cfg["rank"]
    theta, A_ctx = make_ground_truth(d, rank, A_SCALE, seed)
    A = A_ctx if world == "context" else np.zeros_like(A_ctx)
    dev = {k: to_dev(v) for k, v in build_splits(theta, A, n, d, data_seed=seed + 1).items()}
    rows = []
    for name in models:
        h = hp.get(family(name), HP_DEFAULT)
        torch.manual_seed(seed)
        model = build_model(name, d).to(DEVICE)
        row = dict(cfg=cfg["id"], d=d, n=n, rank=rank, world=world, model=name, seed=seed,
                   params=n_params(model), lr=h["lr"], wd=h["wd"])
        row.update(train_model(model, dev["train"], dev["val"], h["lr"], h["wd"]))
        row["nll_in"], row["kl_in"] = eval_split(model, dev["eval_in"])
        row["nll_shift"], row["kl_shift"] = eval_split(model, dev["eval_shift"])
        if measure:
            sf = score_fn_of(model, shuffle=not is_equivariant(name), seed=seed)
            rep, _, _ = probe_report(sf, A, theta, d)
            row.update(rep); row.update(metric_c(sf, A, A_ctx, d))
        row["tag"] = TAG
        rows.append(row)
    return rows

def load_csv(path):
    if not os.path.exists(path): return pd.DataFrame()
    df = pd.read_csv(path)
    return df[df.tag == TAG].reset_index(drop=True) if "tag" in df else pd.DataFrame()

def run_jobs(jobs, path, hp, measure=True):
    # jobs: list of (cfg, world, seed, models). Only models not already saved are trained.
    done = load_csv(path)
    keys = set(zip(done.cfg, done.world, done.seed, done.model)) if len(done) else set()
    t0 = time.time()
    todo = []
    for cfg, world, seed, models in jobs:
        missing = [m for m in models if (cfg["id"], world, seed, m) not in keys]
        if missing: todo.append((cfg, world, seed, missing))
    print(f"{len(jobs) - len(todo)} jobs fully saved, {len(todo)} with models still to run")
    for i, (cfg, world, seed, models) in enumerate(todo):
        rows = run_world(cfg, world, seed, models, hp, measure)
        pd.DataFrame(rows).to_csv(path, mode="a", header=not os.path.exists(path), index=False)
        print(f"[{i + 1}/{len(todo)}] {cfg['id']:<14} {world:<9} seed={seed:<4} ({time.time() - t0:.0f}s)")
    return load_csv(path)

## P1: the probe on functions with a known answer, at d = 6 and d = 16

Same idea as the ladder in E0, plus two additions:

- **V0, a context-free but nonlinear function** ($\theta^\top x_i + \sum_p \sin(x_{ip})$). It has no context effect at all, so the probe must read exactly 0. This checks the claim that fixing the pair cancels the model's own taste for each item.
- Everything is repeated at **d = 16** (rank 8), the largest size in the sweeps, to make sure the probe still works with 256 unknowns.

**Pass looks like:** V0 reads 0; V1 and V2 read cosine 1, size 1, θ cosine 1, fidelity 1; V4 reads fidelity clearly below 1.

In [ ]:
def known_functions(theta, A):
    U_, S_, Vt_ = np.linalg.svd(A)
    Wq, Wk = np.sqrt(S_)[:, None] * U_.T, np.sqrt(S_)[:, None] * Vt_
    def v0(X): return X @ theta + np.sin(X).sum(axis=-1)
    def v1(X): return np.einsum("nd,nsd->ns", theta[None, :] + X.mean(axis=1) @ A.T, X)
    def v2(X):
        q, k = X @ Wq.T, X @ Wk.T
        return X @ theta + np.einsum("nid,njd->nij", q, k).mean(axis=2)
    def v4(X):
        z = np.einsum("nd,nsd->ns", X.mean(axis=1) @ A.T, X)
        return X @ theta + 0.5 * np.tanh(2.0 * z)
    return [("V0 context-free, nonlinear", v0), ("V1 generator's own score", v1),
            ("V2 hand-built attention", v2), ("V4 saturating (not LCL)", v4)]

p1_rows = []
for d, rank in [(D_REF, RANK_REF), (8 if QUICK else 16, 4 if QUICK else 8)]:
    th, A = make_ground_truth(d, rank, A_SCALE, seed=12345)
    for name, fn in known_functions(th, A):
        r, A_hat, th_hat = probe_report(fn, A, th, d)
        r.update(metric_c(fn, A, A, d))
        r.update(rung=name, d=d, max_err_A=np.abs(A_hat - A).max(), max_err_theta=np.abs(th_hat - th).max())
        p1_rows.append(r)
p1 = pd.DataFrame(p1_rows)
cols = ["d", "rung", "fro_hat", "cosine", "scale_ratio", "theta_cos", "fid_A", "inv_rel", "max_err_A", "max_err_theta"]
print(p1[cols].to_string(index=False, float_format=lambda v: f"{v:.4g}"))

## P2: Metric C on the same known functions

The same four functions, read with the rebuilt Metric C (computed in the cell above, shown here).

**Pass looks like:** V0 gives c_ratio = 0 (no movement at all); V1 and V2 give slope 1 and R² 1; V4 gives slope below 1, because the saturating function moves less than the true rule for large changes.

In [ ]:
print(p1[["d", "rung", "c_slope", "c_r2", "c_ratio"]].to_string(index=False, float_format=lambda v: f"{v:.4g}"))

## P3: tuning learning rate and weight decay, by validation NLL only

**What is tuned:** learning rate and weight decay, for each model family, on the reference world (d = 6, rank 2, 40k sessions), 3 seeds.

**What is not tuned:** model size. Sizes stay matched in parameters (mean-pool vs. self-attention), because size is part of the architecture comparison, not a training knob.

**The one rule:** the choice is made on **validation NLL only**. Choosing by recovery of A would use the ground truth, and then the recovery numbers could no longer be trusted.

A warning is printed if the winner sits at the edge of the grid (then the grid should be extended), and if any run hit the epoch budget.

In [ ]:
TUNE_LR    = [1e-2, 3e-2] if QUICK else [3e-3, 1e-2, 3e-2]
TUNE_WD    = [0.0] if QUICK else [0.0, 1e-4, 1e-3]
TUNE_SEEDS = [1] if QUICK else [1, 2, 3]
TUNE_MODELS = ["flat_L2", "flat_L4", "flat_L6", "mean_pool", "self_attn_L1", "self_attn_L6",
               f"bilinear_h{D_REF}", f"softmaxov_h{D_REF}", "oracle"]
TUNE_CSV = os.path.join(OUT_DIR, "p3_tuning.csv")

ref_cfg = dict(id="ref", d=D_REF, n=N_REF, rank=RANK_REF)
prev = load_csv(TUNE_CSV)
done = set(zip(prev.model, prev.lr, prev.wd, prev.seed)) if len(prev) else set()
t0 = time.time()
for seed in TUNE_SEEDS:
    th, A = make_ground_truth(D_REF, RANK_REF, A_SCALE, seed)
    dev = {k: to_dev(v) for k, v in build_splits(th, A, N_REF, D_REF, data_seed=seed + 1).items()}
    for name in TUNE_MODELS:
        for lr in TUNE_LR:
            for wd in TUNE_WD:
                if (name, lr, wd, seed) in done: continue
                torch.manual_seed(seed)
                model = build_model(name, D_REF).to(DEVICE)
                r = dict(model=name, family=family(name), lr=lr, wd=wd, seed=seed, tag=TAG)
                r.update(train_model(model, dev["train"], dev["val"], lr, wd))
                pd.DataFrame([r]).to_csv(TUNE_CSV, mode="a", header=not os.path.exists(TUNE_CSV), index=False)
        print(f"seed {seed}: {name} done ({time.time() - t0:.0f}s)")
tune = load_csv(TUNE_CSV)
print(len(tune), "tuning runs saved")

In [ ]:
tune = tune[tune.family.isin({family(m) for m in TUNE_MODELS})]      # drop v1's old softmax model
grid = tune.groupby(["family", "lr", "wd"]).agg(val_nll=("val_nll", "mean"), sd=("val_nll", "std"),
                                                 best_epoch=("best_epoch", "median"),
                                                 budget_hits=("budget_hit", "sum")).reset_index()
hp, warnings = {}, []
for fam, g in grid.groupby("family"):
    best = g.loc[g.val_nll.idxmin()]
    hp[fam] = dict(lr=float(best.lr), wd=float(best.wd))
    if len(TUNE_LR) > 1 and best.lr in (min(TUNE_LR), max(TUNE_LR)):
        warnings.append(f"{fam}: best lr {best.lr} is at the edge of the grid -> extend the grid")
    if g.budget_hits.sum() > 0:
        warnings.append(f"{fam}: {int(g.budget_hits.sum())} runs hit the epoch budget")
    print(f"\n{fam}  ->  chosen lr={best.lr:g}, wd={best.wd:g}")
    print(g.sort_values("val_nll").head(5).to_string(index=False, float_format=lambda v: f"{v:.5g}"))

with open(HP_PATH, "w") as f: json.dump(hp, f, indent=2)
print("\nsaved", HP_PATH)
print("\n".join(warnings) if warnings else "no tuning warnings")

## P4: the null floor and the effect, across data size and number of features

This is the most important check. For each setting we train the models in **both** worlds:

- **no-effect world (A = 0):** whatever size $\|\hat A\|_F$ shows here is invented. That is the **floor**.
- **context world ($\|A\|_F = 1$):** the real recovery.

From the floor we get the **detection threshold** = floor mean + 2 standard deviations: below that size, a recovered effect cannot be told apart from zero at that setting.

| settings | why |
|---|---|
| d = 6, n = 2.5k, 10k, 40k, 160k | the floor should shrink with data; the smallest n is where it is most likely too big |
| d = 12 and 16, at n = 40k | more unknowns with the same data |
| d = 12 and 16, at n ∝ d² (160k, 285k) | more unknowns, data scaled to match |

20 seeds at the reference setting (they also feed the seed calculation in P6), 10 elsewhere. The 6-layer transformer is included at d = 6. **This is the longest cell** (roughly 2–3 GPU hours); it saves after each job.

In [ ]:
hp = load_hp()
assert hp, "run P3 first (no tuned hyperparameters found)"

if QUICK:
    FLOOR_CFGS = [dict(id="n500_d6", d=6, n=500, rank=2), dict(id="ref", d=6, n=N_REF, rank=2),
                  dict(id="d8_n2k", d=8, n=2000, rank=3)]
    seeds_for = lambda cfg: [101, 202]
else:
    FLOOR_CFGS = [dict(id="n2.5k_d6", d=6, n=2500, rank=2), dict(id="n5k_d6", d=6, n=5000, rank=2),
                  dict(id="n10k_d6", d=6, n=10000, rank=2),
                  dict(id="ref", d=6, n=40000, rank=2), dict(id="n160k_d6", d=6, n=160000, rank=2),
                  dict(id="d12_n40k", d=12, n=40000, rank=4), dict(id="d12_n160k", d=12, n=160000, rank=4),
                  dict(id="d16_n40k", d=16, n=40000, rank=5), dict(id="d16_n285k", d=16, n=285000, rank=5)]
    seeds_for = lambda cfg: list(range(101, 121)) if cfg["id"] == "ref" else list(range(101, 111))

def models_for(cfg):
    if cfg["d"] == 6:
        return ["flat_L2", "flat_L4", "flat_L6", "mean_pool", "self_attn_L1", "self_attn_L6", "oracle"]
    return ["mean_pool", "self_attn_L1", "oracle"]

FLOOR_CSV = os.path.join(OUT_DIR, "p4_floor_map.csv")
jobs = [(cfg, world, s, models_for(cfg)) for cfg in FLOOR_CFGS for s in seeds_for(cfg)
        for world in ["no_effect", "context"]]
fm = run_jobs(jobs, FLOOR_CSV, hp)
print(len(fm), "rows")

In [ ]:
nul, ctx = fm[fm.world == "no_effect"], fm[fm.world == "context"]
floor = nul.groupby(["cfg", "d", "n", "model"]).agg(floor=("fro_hat", "mean"), floor_sd=("fro_hat", "std"),
                                                     c_false=("c_ratio", "mean"), seeds=("seed", "nunique"))
eff = ctx.groupby(["cfg", "d", "n", "model"]).agg(effect=("fro_hat", "mean"), cosine=("cosine", "mean"),
                                                  rel_frob=("rel_frob", "mean"), theta_cos=("theta_cos", "mean"),
                                                  fid_A=("fid_A", "mean"), c_slope=("c_slope", "mean"),
                                                  kl_in=("kl_in", "mean"), kl_shift=("kl_shift", "mean"))
fmap = floor.join(eff).reset_index()
fmap["threshold"] = fmap.floor + 2 * fmap.floor_sd.fillna(0)
fmap["floor/effect"] = fmap.floor / fmap.effect
fmap = fmap.sort_values(["d", "n", "model"])
pd.set_option("display.width", 250)
print(fmap.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
fmap.to_csv(os.path.join(OUT_DIR, "p4_floor_summary.csv"), index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))
sub = fmap[fmap.d == D_REF]
for m, g in sub.groupby("model"):
    axes[0].errorbar(g.n, g.floor, yerr=g.floor_sd, marker="o", capsize=3, label=f"{m}: floor (A=0)")
    axes[0].plot(g.n, g.effect, ls="--", marker="x", alpha=0.6, label=f"{m}: recovered size (true 1)")
axes[0].set_xscale("log"); axes[0].set_xlabel("training sessions"); axes[0].set_ylabel("||A_hat||_F")
axes[0].set_title(f"floor vs. real effect, d={D_REF}"); axes[0].legend(fontsize=7)
sns.barplot(data=fmap[fmap.d > D_REF], x="cfg", y="floor/effect", hue="model", ax=axes[1])
axes[1].axhline(0.25, color="gray", ls=":"); axes[1].set_title("floor / effect at more features (lower is better)")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "p4_floor_map.png"), dpi=150); plt.show()

## P4b: the professor's question: does a plain, deeper network do just as well?

All seven models at d = 6, side by side, from the same runs as P4. Read it as a ladder from "nothing built in" to "everything built in":

| model | what it is told about the problem |
|---|---|
| `flat_L2/4/6` | nothing: it sees all 5 items as one long list and must learn that order doesn't matter and that the average matters |
| `self_attn_L1/L6` | that the input is a **set** (order-free); it must learn how to combine the items |
| `mean_pool` | the set **and** the exact summary LCL uses (the average $y_S$) |
| `oracle` | the full formula |

**How to read it.** If the flat MLP gets close to self-attention as depth grows, depth alone is enough and attention adds nothing special for this problem. If it stays clearly worse at every depth, the set structure is what matters. The flat MLP's gap to self-attention, and self-attention's gap to mean-pool, are the two numbers that answer the question.

In [ ]:
CMP_MODELS = ["flat_L2", "flat_L4", "flat_L6", "self_attn_L1", "self_attn_L6", "mean_pool", "oracle"]
cmp_ = fmap[(fmap.d == D_REF) & fmap.model.isin(CMP_MODELS)]
for met, title in [("effect", "recovered size of A (true 1)"), ("cosine", "direction of A (cosine, 1 = perfect)"),
                   ("rel_frob", "overall error in A (rel_frob, 0 = perfect)"), ("fid_A", "fidelity (how LCL-shaped)"),
                   ("theta_cos", "theta direction (cosine)"), ("kl_in", "KL to the true choice probabilities (lower = better)"),
                   ("floor", "false-positive floor on A = 0 data")]:
    print(f"\n{title}")
    print(cmp_.pivot_table(index="n", columns="model", values=met).reindex(columns=CMP_MODELS).round(3).to_string())

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for ax, met, lab in zip(axes, ["rel_frob", "cosine", "kl_in"], ["rel_frob (lower = better)", "cosine (higher = better)", "KL (lower = better)"]):
    for m in CMP_MODELS:
        g = cmp_[cmp_.model == m].sort_values("n")
        ax.plot(g.n, g[met], marker="o", ls="--" if m.startswith("flat") else "-", label=m)
    ax.set_xscale("log"); ax.set_xlabel("training sessions"); ax.set_title(lab)
    if met == "kl_in": ax.set_yscale("log")
axes[0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "p4b_plain_network.png"), dpi=150); plt.show()

## P5: do the new attention models behave as the theory says?

The rank experiment (E3) rests on one claim: **a bilinear attention head of size h can only represent the best rank-h part of A**, so its recovery should sit exactly on the Eckart–Young ceiling

$$\text{best possible cosine with rank } h = \sqrt{\frac{\sum_{i\le h}\sigma_i^2}{\sum_i\sigma_i^2}}$$

($\sigma_i$ are the singular values of the true A). We test that here on a small grid before building E3 on it: true rank 2 and 6, head size h = 1, 2, 4, 6, for both the bilinear model and the (fixed) softmax head. The softmax head's context matrix is $W_O^\top W_V$, which also has rank at most h, so the same ceiling applies to it.

**Pass looks like:** for h below the true rank, the bilinear model's cosine is close to the ceiling (from below); for h at or above the rank, it matches the oracle. For the softmax head there is no pass/fail: whether it reaches the ceiling too is exactly the question E3 asks.

In [ ]:
ARCH_SEEDS = [101] if QUICK else [101, 202, 303, 404, 505]
H_LIST = [1, 2, D_REF] if QUICK else [1, 2, 4, D_REF]
ARCH_CFGS = [dict(id=f"arch_r{r}", d=D_REF, n=N_REF, rank=r) for r in [2, D_REF]]
arch_models = [f"{fam}_h{h}" for fam in ["bilinear", "softmaxov"] for h in H_LIST] + ["oracle"]
ARCH_CSV = os.path.join(OUT_DIR, "p5_architectures.csv")
arch = run_jobs([(c, "context", s, arch_models) for c in ARCH_CFGS for s in ARCH_SEEDS], ARCH_CSV, hp)

def ey_ceiling(rank, seed, h):
    s = np.linalg.svd(make_ground_truth(D_REF, rank, A_SCALE, seed)[1], compute_uv=False)
    return float(np.sqrt((s[:h] ** 2).sum() / (s ** 2).sum()))

att = arch[arch.model.str.startswith(("bilinear_h", "softmaxov_h"))].copy()      # v1's broken softmax_h6 is left out
att["family"] = att.model.str.split("_h").str[0]
att["h"] = att.model.str.split("_h").str[1].astype(int)
att["ceiling"] = [ey_ceiling(r, s, h) for r, s, h in zip(att["rank"], att.seed, att.h)]
att["gap"] = att.cosine - att.ceiling
print(att.groupby(["family", "rank", "h"])[["cosine", "ceiling", "gap", "fid_A", "kl_in"]].mean().round(4).to_string())
print("\noracle (mean over seeds):")
print(arch[arch.model == "oracle"].groupby("rank")[["cosine", "scale_ratio", "fid_A", "c_slope", "kl_in"]].mean().round(4).to_string())
bil, sov = att[att.family == "bilinear"], att[att.family == "softmaxov"]

## P6: how many seeds does each comparison need?

From the 20 reference seeds in P4 we compute, for each pair of models, the spread of the **paired** difference (same seed, same data). The number of seeds needed to detect a difference Δ with 80% power at the 5% level is about $(2.8\,\sigma_{\text{diff}}/\Delta)^2$.

Read the table as: "to tell self-attention from mean-pool when they differ by 0.03 in rel_frob, we need N seeds." Pick the smallest Δ that would actually change a conclusion in the thesis, and use that row.

In [ ]:
ref = ctx[ctx.cfg == "ref"]
PAIRS = [("self_attn_L1", "mean_pool"), ("self_attn_L1", "oracle"), ("mean_pool", "oracle"),
         ("self_attn_L6", "self_attn_L1"), ("self_attn_L1", "flat_L4"), ("flat_L6", "flat_L2")]
DELTAS = {"rel_frob": [0.02, 0.03, 0.05], "scale_ratio": [0.02, 0.03, 0.05], "cosine": [0.005, 0.01, 0.02],
          "c_slope": [0.02, 0.03, 0.05], "kl_in": [0.0005, 0.001, 0.002]}
rows = []
for a, b in PAIRS:
    pa, pb = ref[ref.model == a].set_index("seed"), ref[ref.model == b].set_index("seed")
    for met, deltas in DELTAS.items():
        diff = (pa[met] - pb[met]).dropna()
        if len(diff) < 3: continue
        sd = diff.std()
        r = dict(pair=f"{a} - {b}", metric=met, mean_diff=diff.mean(), sd_diff=sd, n_now=len(diff))
        for D_ in deltas: r[f"seeds for {D_:g}"] = int(np.ceil((2.8 * sd / D_) ** 2))
        rows.append(r)
power = pd.DataFrame(rows)
if power.empty:
    print("fewer than 3 reference seeds saved: nothing to compute yet")
else:
    print(power.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
power.to_csv(os.path.join(OUT_DIR, "p6_seed_requirements.csv"), index=False)

## P7: how long will the large runs take?

Measured seconds per training run, from P4, and a rough budget for the two experiments that answer the professor's question first (E1 headline, E2 data-size). Time grows roughly in proportion to the number of training sessions, so for sizes not measured here we scale from the nearest measured one.

In [ ]:
t = fm.groupby(["d", "n", "model"]).train_seconds.median().reset_index()
print(t.pivot_table(index=["d", "n"], columns="model", values="train_seconds").round(1).to_string())

def sec_per_run(model, n, d=D_REF):
    g = t[(t.model == model) & (t.d == d)]
    if g.empty: g = t[t.model == model]
    near = g.iloc[(np.log(g.n) - np.log(n)).abs().argmin()]
    return near.train_seconds * n / near.n + 5            # + ~5 s for probe and metric C

E1_MODELS = ["mean_pool", "self_attn_L1", "oracle"]
E2_N = [2500, 5000, 10000, 20000, 40000, 80000, 160000, 320000]
e1 = 50 * 2 * sum(sec_per_run(m, 40000) for m in E1_MODELS)                  # 50 seeds x 2 worlds
e2 = 20 * 2 * sum(sec_per_run(m, n) for m in E1_MODELS for n in E2_N)       # 20 seeds x 2 worlds
print(f"\nE1 (50 seeds, both worlds): ~{e1 / 3600:.1f} GPU-hours if run one after another")
print(f"E2 (20 seeds, both worlds, 8 data sizes): ~{e2 / 3600:.1f} GPU-hours if run one after another")
print("Divide by the number of jobs running in parallel on the cluster.")

## P8: go / no-go

Every check from P1–P6 in one table. **FAIL** means stop and fix before launching. **CHECK** means not broken, but a decision is needed (for example, an effect size that cannot be resolved at some setting). The thresholds are written in the cell so you can see and change them.

At the end, everything the cluster runs need is written to `preflight_summary.json`: tuned hyperparameters, the detection threshold per setting and model, and the seed requirements.

In [ ]:
checks = []
def add(name, ok, detail, fail_is_check=False):
    checks.append(dict(check=name, status="PASS" if ok else ("CHECK" if fail_is_check else "FAIL"), detail=detail))

# P1: probe
for d in p1.d.unique():
    q = p1[p1.d == d].set_index("rung")
    v0 = q.loc["V0 context-free, nonlinear"]
    add(f"P1 d={d}: context-free model reads A_hat = 0", v0.fro_hat < 1e-8, f"||A_hat|| = {v0.fro_hat:.2e}")
    for r in ["V1 generator's own score", "V2 hand-built attention"]:
        e = max(q.loc[r].max_err_A, q.loc[r].max_err_theta)
        add(f"P1 d={d}: {r[:2]} recovers A and theta exactly", e < 1e-6, f"max error {e:.2e}")
    f4 = q.loc["V4 saturating (not LCL)"].fid_A
    add(f"P1 d={d}: fidelity notices a non-LCL function", f4 < 0.99, f"V4 fidelity {f4:.3f}")
    # P2: metric C
    v1 = q.loc["V1 generator's own score"]
    add(f"P2 d={d}: Metric C exact on the true rule", abs(v1.c_slope - 1) < 1e-6 and v1.c_r2 > 1 - 1e-9,
        f"slope {v1.c_slope:.6f}, R2 {v1.c_r2:.6f}")
    add(f"P2 d={d}: Metric C reads 0 for a context-free model", v0.c_ratio < 1e-8, f"c_ratio {v0.c_ratio:.2e}")

# P3: tuning
add("P3: tuning winners inside the grid, no budget hits", not warnings, "; ".join(warnings) or "ok", fail_is_check=True)

# P4: training budget and floor
hits = int(fm.budget_hit.sum())
add("P4: no run hit the epoch budget", hits == 0, f"{hits} of {len(fm)} runs", fail_is_check=True)
bad = fmap[fmap["floor/effect"] > 0.25]
add("P4: floor below 25% of the effect everywhere", bad.empty,
    "ok" if bad.empty else "too noisy at: " + ", ".join(f"{c}/{m}" for c, m in zip(bad.cfg, bad.model)),
    fail_is_check=True)
for c, g in fmap.groupby("cfg"):
    o = g[g.model == "oracle"]
    for m in g.model:
        if m == "oracle" or o.empty: continue
        r = g[g.model == m].iloc[0]
        lim = o.floor.iloc[0] + 2 * o.floor_sd.iloc[0]
        add(f"P4 {c}: {m} floor not above the oracle's noise", r.floor <= lim,
            f"{r.floor:.3f} vs oracle {o.floor.iloc[0]:.3f} (+2sd = {lim:.3f})", fail_is_check=True)

# P5: Eckart-Young
below = bil[bil.h < bil["rank"]].groupby(["rank", "h"]).gap.mean()
add("P5: bilinear model sits on the Eckart-Young ceiling when h < rank", (below.abs() < 0.03).all(),
    ", ".join(f"r{r}/h{h}: {g:+.3f}" for (r, h), g in below.items()), fail_is_check=True)
orc = arch[arch.model == "oracle"].groupby("rank").cosine.mean()
above = bil[bil.h >= bil["rank"]].groupby("rank").cosine.mean()
add("P5: bilinear model matches the oracle when h >= rank", ((orc - above).abs() < 0.02).all(),
    ", ".join(f"r{r}: {above[r]:.3f} vs {orc[r]:.3f}" for r in above.index), fail_is_check=True)
sov_below = sov[sov.h < sov["rank"]].groupby(["rank", "h"]).gap.mean()
sov_above = sov[sov.h >= sov["rank"]].groupby("rank").cosine.mean()
add("P5 (informative): softmax head vs the Eckart-Young ceiling when h < rank", (sov_below.abs() < 0.03).all(),
    ", ".join(f"r{r}/h{h}: {g:+.3f}" for (r, h), g in sov_below.items()), fail_is_check=True)
add("P5 (informative): softmax head vs the oracle when h >= rank", ((orc - sov_above).abs() < 0.02).all(),
    ", ".join(f"r{r}: {sov_above[r]:.3f} vs {orc[r]:.3f}" for r in sov_above.index), fail_is_check=True)

go = pd.DataFrame(checks)
for c in checks:
    print(f"{c['status']:<6} {c['check']:<68} {c['detail']}")
print("\nFAIL:", (go.status == "FAIL").sum(), "| CHECK:", (go.status == "CHECK").sum(), "| PASS:", (go.status == "PASS").sum())

summary = dict(tag=TAG, hyperparameters=hp,
               detection_threshold={f"{r.cfg}|{r.model}": round(float(r.threshold), 4) for r in fmap.itertuples()},
               seed_requirements=power.to_dict(orient="records"),
               checks=checks)
with open(os.path.join(OUT_DIR, "preflight_summary.json"), "w") as f: json.dump(summary, f, indent=2, default=float)
print("saved", os.path.join(OUT_DIR, "preflight_summary.json"))

### The frozen settings file (a draft for the cluster runs)

Everything the cluster script will read, in one place, so the large runs use exactly what was checked here. The model lists are a draft: we finalise them after looking at P4b and P5 together.

In [ ]:
BOTH = ["no_effect", "context"]
final_config = dict(
    tag=TAG, status="draft: review before the cluster script uses it",
    hyperparameters=hp,
    protocol=dict(optimizer="AdamW, full batch (chunked)", max_epochs=MAX_EPOCHS, patience=PATIENCE, chunk=CHUNK,
                  early_stopping="validation NLL only", n_val=N_VAL, n_eval=N_EVAL, n_shift=N_SHIFT,
                  slate=SLATE, A_scale=A_SCALE, transformer="pre-LN, d_model 32, 2 heads, FFN 64, no dropout"),
    measurement=dict(probe_pairs=f"{PROBE_K_PER_D} x d", probe_slates_per_pair=PROBE_N, bootstrap=N_BOOT,
                     metric_c_trials=C_TRIALS, metric_c_reported="c_slope (+ c_ratio on A=0 data)",
                     flat_mlp_probe="item order shuffled"),
    experiments=dict(
        E1=dict(question="headline comparison", d=6, rank=2, n=40000, seeds=50, worlds=BOTH,
                models=["flat_L2", "flat_L4", "flat_L6", "self_attn_L1", "self_attn_L6", "mean_pool",
                        f"bilinear_h{D_REF}", f"softmaxov_h{D_REF}", "oracle"]),
        E2=dict(question="how fast does error shrink with data", d=6, rank=2,
                n=[2500, 5000, 10000, 20000, 40000, 80000, 160000, 320000], seeds=20, worlds=BOTH,
                models=["flat_L4", "self_attn_L1", "mean_pool", "oracle"]),
        E6=dict(question="identification: same data, different initialisations", d=6, rank=2, n=40000,
                datasets=10, inits_per_dataset=20, worlds=["context"],
                models=["self_attn_L1", "mean_pool", "oracle"])),
    detection_threshold=summary["detection_threshold"])
with open(os.path.join(OUT_DIR, "final_config.json"), "w") as f: json.dump(final_config, f, indent=2, default=float)
print("saved", os.path.join(OUT_DIR, "final_config.json"))

## What to send back

1. The **go / no-go table** from P8.
2. The tables from **P4b** (the professor's question).
3. The **Eckart–Young** table from P5 (now with the softmax head).
4. The **seed table** from P6.

With those we settle the model list and seed counts, and only then write the cluster script.